# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the [FAIR² Mass Spectrometry](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL (JSON-LD).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# View top-level metadata fields
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n")
print(f"Published: {meta.datePublished}\n")
print(f"License: {meta.license}\n")
print(f"Keywords: {meta.keywords}\n")

## 2. Data Overview

Review available record sets, fields, and their `@id`s.

> In Croissant, each record set, field/column is referenced by their unique `@id` (IRI).

In [ ]:
# List all record sets with their @id and name
print("Available record sets:")
for rs in dataset.metadata.record_sets:
    print(f"  ID: {rs.id} | name: {rs.name}")

# For each record set, list its fields (columns) with their @id
print("\nRecord sets and their fields/columns:")
for rs in dataset.metadata.record_sets:
    print(f"\nRecord Set: {rs.name}\n  ID: {rs.id}")
    for f in rs.fields:
        print(f"    Field: {f.name:30} @id: {f.id}    (dataType: {f.data_type})")

## 3. Data Extraction

We load records from one or more record sets (tables), referencing them by their `@id`.

Let's load all records from the main record set into pandas DataFrames for analysis.

In [ ]:
# Compile all record sets' @id
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print("List of record set @id's:")
pprint.pprint(record_set_ids)

# We'll load all tables into a dict of DataFrames, referenced by the record set's @id
dataframes = {}

for record_set_id in record_set_ids:
    print(f"\nLoading: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    df = dataframes[record_set_id]
    print(f"  Loaded {len(df)} records. Columns:")
    print(df.columns.tolist())

# Let's display the first 5 rows from the main table (first record set found):
main_rs_id = record_set_ids[0]
display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Let's filter and normalize a numeric field. Each field/column is referenced by its `@id`.

For demonstration, we select relevant numeric columns.

In [ ]:
main_rs = dataset.metadata.record_sets[0]
print(f"Using main record set: {main_rs.id}\n")

# Find numeric fields in this record set, print their name and @id
numeric_field_ids = [f.id for f in main_rs.fields if f.data_type in ('Number', 'Float', 'Integer')]
print("Numeric fields (@id):")
for f in main_rs.fields:
    if f.data_type in ('Number', 'Float', 'Integer'):
        print(f"  {f.name:30} {f.id}")

# Choose a numeric field for demonstration
if numeric_field_ids:
    numeric_field_id = numeric_field_ids[0]  # Take the first one
else:
    raise ValueError("No numeric fields found in record set!")

print(f"Using numeric field: {numeric_field_id}")

df = dataframes[main_rs.id]

if numeric_field_id not in df.columns:
    raise RuntimeError(f"Field {numeric_field_id} not found in DataFrame columns. Available: {df.columns.tolist()}")

# Remove rows with missing/NaN numeric values
df_numeric = df.dropna(subset=[numeric_field_id])
# Attempt conversion to float if not numeric
df_numeric[numeric_field_id] = pd.to_numeric(df_numeric[numeric_field_id], errors='coerce')

threshold = df_numeric[numeric_field_id].quantile(0.75)  # Use 75th percentile as example threshold
filtered_df = df_numeric[df_numeric[numeric_field_id] > threshold]
print(f"\nFiltered records with {numeric_field_id} > {threshold:.3f} (top quartile): {len(filtered_df)} rows")
display(filtered_df.head(3))

# Normalization (z-score)
filtered_df[numeric_field_id + '_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head(5))

# Determine candidate group-by fields (categorical string columns with not too many unique values)
group_fields = [f.id for f in main_rs.fields if f.data_type in ('Text',) and df[f.id].nunique() < 10]
print("\nSuggested grouping fields (less than 10 unique values):", group_fields)
if group_fields:
    group_field = group_fields[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
    print(f"\nMean {numeric_field_id} grouped by {group_field}:")
    display(grouped_df.head())

## 5. Visualization

Let's plot the distribution of our selected numeric column, and if suitable, group means.

In [ ]:
import matplotlib.pyplot as plt

# Histogram of the numeric field
plt.figure(figsize=(7,4))
df_numeric[numeric_field_id].hist(bins=40, alpha=0.7)
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.title(f"Histogram of {numeric_field_id}")
plt.show()

# Grouped means plot
if group_fields:
    plt.figure(figsize=(8,4))
    grouped_df.sort_values(ascending=False).plot(kind='bar')
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.title(f"Mean {numeric_field_id} by {group_field}")
    plt.show()

## 6. Conclusion

In this notebook, we've demonstrated how to load and explore a mass spectrometry dataset annotated in Croissant FAIR schema using `mlcroissant`.
- The dataset fields and tables are referenced by their canonical `@id` as defined in the schema.
- Simple EDA and visualization was applied to a numeric column and grouped means.

You can modify the code to select different fields, try other record sets, or perform more advanced processing as needed.

For full documentation, visit [`mlcroissant` docs](https://mlcommons.github.io/croissant).